Đọc dữ liệu

In [1]:

import os
import numpy as np
import pandas as pd

# Đường dẫn dữ liệu giống Decision Tree
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc dữ liệu
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước dữ liệu
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

print("\n--- 5 dòng đầu tiên ---")
print(diamonds.head())

print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
print(f"Cut: {diamonds['cut'].unique()}")
print(f"Color: {diamonds['color'].unique()}")
print(f"Clarity: {diamonds['clarity'].unique()}")

print("\n--- Thống kê mô tả ---")
print(diamonds.describe())


Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.core.frame.DataFrame'>
Index: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  object 
 2   color    53943 non-null  object 
 3   clarity  53943 non-null  object 
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I 

Tiền xử lý dữ liệu

In [3]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Mã hóa thứ tự cho các thuộc tính phân loại
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

# Khởi tạo bộ mã hóa
encoder = OrdinalEncoder(
    categories=[cut_categories, color_categories, clarity_categories]
)

# Mã hóa các cột dữ liệu dạng chữ thành số
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 3: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")
print(diamonds.head())



Kích thước tập dữ liệu sau tiền xử lý: (53943, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [4]:

from sklearn.model_selection import train_test_split

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])

# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu train test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# In ra kết quả chia dữ liệu
print(f"Có {X_train.shape[0]} mẫu huấn luyện và {X_test.shape[0]} mẫu kiểm tra")


Có 43154 mẫu huấn luyện và 10789 mẫu kiểm tra


Training Random Forest

In [6]:

# Import mô hình Random Forest hồi quy
from sklearn.ensemble import RandomForestRegressor

# Import các hàm đánh giá
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import thư viện đo thời gian
import time

# Khởi tạo mô hình với bộ tham số mặc định
model = RandomForestRegressor(
    criterion="squared_error",
    random_state=42
)

# Bắt đầu đo thời gian huấn luyện
start_time = time.perf_counter()

# Huấn luyện mô hình trên tập Train
model.fit(X_train, y_train)

# Tính thời gian huấn luyện
training_time = time.perf_counter() - start_time

# Dự đoán trên tập Train
y_pred_train = model.predict(X_train)

# Dự đoán trên tập Test
y_pred_test = model.predict(X_test)

# Tính MAE trên Train
mae_train = mean_absolute_error(y_train, y_pred_train)

# Tính MSE trên Train
mse_train = mean_squared_error(y_train, y_pred_train)

# Tính RMSE trên Train
rmse_train = np.sqrt(mse_train)

# Tính R² trên Train
r2_train = r2_score(y_train, y_pred_train)

# Tính MAE trên Test
mae_test = mean_absolute_error(y_test, y_pred_test)

# Tính MSE trên Test
mse_test = mean_squared_error(y_test, y_pred_test)

# Tính RMSE trên Test
rmse_test = np.sqrt(mse_test)

# Tính R² trên Test
r2_test = r2_score(y_test, y_pred_test)

# In các chỉ số đánh giá trên Train
print(f"MAE Train: {mae_train:.4f}")
print(f"MSE Train: {mse_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Train: {r2_train:.4f}")

# In các chỉ số đánh giá trên Test
print(f"MAE Test: {mae_test:.4f}")
print(f"MSE Test: {mse_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")
print(f"R² Test: {r2_test:.4f}")

# In thời gian huấn luyện
print(f"Thời gian Train: {training_time:.4f} giây")


MAE Train: 99.7111
MSE Train: 41342.8746
RMSE Train: 203.3295
R² Train: 0.9974
MAE Test: 261.6055
MSE Test: 284116.8610
RMSE Test: 533.0261
R² Test: 0.9818
Thời gian Train: 12.5539 giây
